# ScreamingFace · quickstart

Connect three model providers, combine their models into one Fusion, evaluate it on five real GPQA
Diamond questions, and compare the Fusion with its strongest member.

This is the shortest supported path: **connect → compose → evaluate → compare**. Its core
evaluation path remains **compose → evaluate → compare**. Evaluation uses real model responses
through the configured ScreamingFace engine; it never substitutes an offline result.

## Before you run it

Start the local development stack from the repository root:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

The local AI Gateway starts with an empty provider profile store. The first cell opens the
ScreamingFace provider panel, which stores provider credentials through the engine. If a selected
provider is disconnected, evaluation raises one actionable `ConnectionRequiredError` before model
calls; provider authentication never becomes repeated per-case failures.

### Gemini compatibility · July 2026

Some newly created Google API projects may receive `model no longer available` for Gemini 2.5
even when their quota dashboard displays Gemini 2.5 limits. The local AI Gateway used by this
notebook does not yet register Google's recommended `gemini-3.5-flash` or
`gemini-3.1-pro-preview` replacements. If that happens, replace the Gemini member below with
another connected model advertised by `sf.models.list()`.

Hugging Face does not provide Gemini through this integration. The forthcoming Hugging Face route
is for open models such as DeepSeek and GLM through pinned inference providers; Gemini 3 still
requires explicit AI Gateway support.

GPQA is fetched by the local ScreamingFace engine. Accept the dataset terms and export a Hugging
Face token in the terminal that starts the stack:

```bash
export HF_TOKEN=hf_...
./dev.sh restart
```

The SDK sends one URL4 HTTP request. Inside that graph, the engine loads the first five cases and
makes 15 model calls: three Fusion members per question. Majority vote, exact-choice grading, and
mean aggregation are engine routes and make no additional provider calls.

## 1 · Connect

In [1]:
import screamingface as sf

sf.connect()

PanelWidget(children=(HTML(value='<style>\n.sf-ui {\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#ef…

Connect each provider used below. The panel sends credentials only to the configured
ScreamingFace engine and shows the engine origin before you act.

## 2 · Compose

In [2]:
fusion = sf.Fusion(
    "frontier-trio",
    members=[
        "codex/gpt-5.5",
        "codex/gpt-5.5",
        "claude/sonnet-4.6",
    ],
    reducer=sf.reducers.MajorityVote(),
)

fusion

Fusion(name='frontier-trio', members=(Model(name='codex/gpt-5.5', model='codex/gpt-5.5', prompt='Answer the question.'), Model(name='codex/gpt-5.5', model='codex/gpt-5.5', prompt='Answer the question.'), Model(name='claude/sonnet-4.6', model='claude/sonnet-4.6', prompt='Answer the question.')), reducer=MajorityVote())

Each member answers the same multiple-choice question. `MajorityVote` selects the
most common exact answer and breaks a tie by stable member order. Fusion construction is local and
does not call a model.

## 3 · Evaluate

In [3]:
benchmark = sf.benchmarks.load("gpqa@1")
report = benchmark.evaluate(fusion, first=5)

HTML(value='<style>\n.sf-ui {\n  --sf-bg:#ffffff;--sf-surface:#f6f6f7;--sf-surface-2:#efeff1;\n  --sf-ink:#161…

`load(...)` reads the benchmark manifest advertised by the configured engine; it
does not download questions. `benchmark.evaluate(...)` then compiles the benchmark, stable
`first=5` slice, Fusion, grader, and aggregator into one reproducible URL4 expression. The engine
executes that graph and streams dataset loading, model activity, completed case grading, and
aggregation before returning one validated report. The case counter advances only after a real
grader result; it is not a time estimate. Missing work remains an explicit failure and is never
silently scored as zero. Pass `progress=False` to hide the compact live status, or `progress=True`
to force it outside notebooks.

## 4 · Compare

In [4]:
report

Report(recipe_name='frontier-trio', benchmark_id='gpqa@1', status='complete', scored=5/5, score=1.000, baseline=1.000, gain=+0.000, failures=0, skipped=0)

Read `gain` first:

- `score` is the Fusion's accuracy across the successfully paired cases;
- `baseline` is the best individual member's accuracy on those same cases; and
- `gain` is `score - baseline`.

A positive gain means the combination outperformed every member on the evaluated cases. A strong
score with zero gain means the Fusion matched, but did not improve on, its strongest member.